# Local setup / data audit
Run `powershell -ExecutionPolicy Bypass -File ASR/local/setup.ps1` from the workspace first. GPU work runs in a separate process and releases memory on exit. Default budget: 4.5 decimal GB total estimated VRAM; duty cycle 0.70 is pacing, not a hardware utilization cap.

In [1]:
from pathlib import Path
import sys, json
here = Path.cwd().resolve()
PROJECT = next((p for p in [here, *here.parents] if (p/'asr/config.py').is_file()), None)
if PROJECT is None:
    PROJECT = next((p/'ASR' for p in [here, *here.parents] if (p/'ASR/asr/config.py').is_file()), None)
assert PROJECT is not None, 'Open notebook from the project workspace'
PYTHON = PROJECT.parent/'.venv-asr/Scripts/python.exe'
DATA = PROJECT/'data'
RAW_DATA = PROJECT.parent/'dataset/ASR'
assert PYTHON.is_file(), 'Run ASR/local/setup.ps1 first'
sys.path.insert(0, str(PROJECT/'local'))
from notebook_process import run_cli as _run_cli
def run_cli(arguments):
    return _run_cli(PROJECT, PYTHON, arguments)
RUNNER = PROJECT/'scripts/local_run.py'
print('Project:', PROJECT)
print('Python:', PYTHON)
print('Ready. Run the specific task cell; model jobs do not start in this setup cell.')


Project: D:\1.FPT_FALL26\DPL - Deep Learning\ASR
Python: D:\1.FPT_FALL26\DPL - Deep Learning\.venv-asr\Scripts\python.exe
Ready. Run the specific task cell; model jobs do not start in this setup cell.


In [2]:
run_cli([RUNNER, 'check'])

[START] "D:\1.FPT_FALL26\DPL - Deep Learning\.venv-asr\Scripts\python.exe" -u "D:\1.FPT_FALL26\DPL - Deep Learning\ASR\scripts\local_run.py" check
[LOG] D:\1.FPT_FALL26\DPL - Deep Learning\ASR\local\logs\20261018_001935_882651.log
[INIT] mode=check experiment=B0_phowhisper_small_zeroshot; importing PyTorch/ASR libraries...
{
  "python": "D:\\1.FPT_FALL26\\DPL - Deep Learning\\.venv-asr\\Scripts\\python.exe",
  "cuda": true,
  "versions": {
    "torch": "2.6.0+cu124",
    "transformers": "4.46.3",
    "peft": "0.13.2",
    "accelerate": "1.1.1",
    "datasets": "3.1.0"
  },
  "missing_data": [],
  "profile": {
    "total_gpu_budget_gb": 4.5,
    "gpu_duty_cycle": 0.7,
    "host_ram_budget_gb": 0,
    "save_steps": 100,
    "checkpoint_interval_seconds": 600,
    "save_total_limit": 2,
    "batch_size": 1,
    "grad_accum": 32,
    "num_workers": 0
  },
  "local_v1_final": false,
  "system_ram_total_gb": 16.989728768,
  "system_ram_available_gb": 2.829258752,
  "gpu_total_gb": 6.43878092

## Prepare all four datasets
RUN_PREPARE defaults to False. Alternatively copy prepared parquet files and registry.json from Colab into ASR/data. Existing raw datasets are never moved or deleted.

In [3]:
RUN_PREPARE = False
if RUN_PREPARE:
    run_cli([PROJECT/'scripts/prepare_data.py', '--raw-dir', RAW_DATA, '--out-dir', DATA])

In [4]:
import pyarrow.parquet as pq
for source in ['VIVOS','FLEURS','VietMed','ViMedCSS']:
    for split in ['train','dev','test']:
        p = DATA/source/f'{split}.parquet'
        print(source, split, pq.ParquetFile(p).metadata.num_rows if p.is_file() else 'MISSING')

VIVOS train 11476
VIVOS dev 184
VIVOS test 760
FLEURS train 2994
FLEURS dev 361
FLEURS test 857
VietMed train 2773
VietMed dev 2912
VietMed test 3437
ViMedCSS train 11832
ViMedCSS dev 1714
ViMedCSS test 1614


## Import Colab artifacts
Copy the entire V1 `final` directory into `ASR/experiments/V1_phowhisper_small_lora/final`. Copy Colab reports into `ASR/local/imports/reports`, full audio into `ASR/local/imports/original_audio`. Paths embedded in imported CSV files still need remapping. `final` is an adapter for inference; optimizer resume requires checkpoint directories as well. Local runs never overwrite imported final.